# S01 — Phase 1 input-configuration audit (VAL)

Reproduces the validation diagnostics used to compare the Phase 1 multisensor input configurations reported in Appendix A.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling\Writing_Article\1_Article\Kimi\Reproducibility\B0_B5_VAL_Audit")
TABLE_DIR = ROOT / "tables"
FIG_DIR = ROOT / "figures"
ARTICLE_FIG_DIR = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling\Writing_Article\1_Article\Kimi\Images\Scenario_Best_Model")
FIG_DIR.mkdir(parents=True, exist_ok=True)
ARTICLE_FIG_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = Path(r"E:\SAFI\SAFI_2\Results_Article_Graph\article_outputs\tables\01_all_discovered_metrics_unique_nearest.csv")
EXPECTED_SHA256 = "c55a0c4bc8aa9381f6b2cb5a97aecabb0f6a400a4b217e644f2a46429842055e"
OBS_MEAN = 9.651921366634731
OBS_STD = 8.194132010097208
N_VAL = 7833

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file(), SOURCE
assert sha256(SOURCE) == EXPECTED_SHA256, "La source VAL archivee a change."

metrics = pd.read_csv(TABLE_DIR / "b0_b5_val_metrics.csv")
mse = pd.read_csv(TABLE_DIR / "b0_b5_val_mse_decomposition.csv")
assert metrics["configuration"].tolist() == [f"B{i}" for i in range(6)]
assert metrics["n_val"].eq(N_VAL).all()
assert mse["n_val"].eq(N_VAL).all()
assert np.allclose(mse["obs_std"], OBS_STD, atol=1e-12, rtol=0)

metrics["obs_mean"] = OBS_MEAN
metrics["pred_mean"] = OBS_MEAN + metrics["bias_m"]
metrics["beta_mean_ratio"] = metrics["pred_mean"] / OBS_MEAN
metrics["alpha_std_ratio"] = metrics["std_ratio"]
metrics["kge_2009"] = 1 - np.sqrt(
    (metrics["corr_r"] - 1) ** 2
    + (metrics["alpha_std_ratio"] - 1) ** 2
    + (metrics["beta_mean_ratio"] - 1) ** 2
)
metrics["r_squared"] = metrics["corr_r"] ** 2
metrics.to_csv(TABLE_DIR / "b0_b5_val_metrics_with_kge.csv", index=False)

provenance = {
    "split": "VAL",
    "support": "common unique-nearest",
    "n": N_VAL,
    "observed_mean": OBS_MEAN,
    "observed_std": OBS_STD,
    "observed_source": str(SOURCE),
    "observed_source_sha256": EXPECTED_SHA256,
    "formula": "KGE2009=1-sqrt((r-1)^2+(alpha-1)^2+(beta-1)^2)",
}
(TABLE_DIR / "b0_b5_val_kge_provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)
metrics[["configuration", "n_val", "mae_m", "bias_m", "corr_r",
         "r_squared", "slope", "alpha_std_ratio", "beta_mean_ratio",
         "kge_2009"]]


In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

colors = ["#1675ad", "#e69f00", "#009e73", "#cc79a7", "#d04a32", "#56b4e9"]
markers = ["o", "s", "^", "D", "P", "X"]
fig, ax = plt.subplots(figsize=(8.2, 6.4))

for row, color, marker in zip(metrics.itertuples(), colors, markers):
    size = 130 if row.configuration == "B4" else 95
    edge = "black" if row.configuration == "B4" else "white"
    ax.scatter(row.beta_mean_ratio, row.alpha_std_ratio, s=size,
               marker=marker, color=color, edgecolor=edge, linewidth=1.1,
               zorder=4)
    dx, dy = {
        "B0": (0.004, 0.006), "B1": (0.003, 0.003),
        "B2": (-0.010, 0.005), "B3": (-0.010, -0.009),
        "B4": (0.003, 0.004), "B5": (0.003, -0.007),
    }[row.configuration]
    ax.text(row.beta_mean_ratio + dx, row.alpha_std_ratio + dy,
            row.configuration,
            color=color, fontsize=9.5,
            fontweight="bold" if row.configuration == "B4" else "normal")

ax.axhline(1, color="0.35", ls="--", lw=1.1)
ax.axvline(1, color="0.35", ls="--", lw=1.1)
ax.scatter([1], [1], marker="*", s=330, color="black", zorder=5)
ax.text(0.996, 0.990, "Ideal", ha="right", va="top", fontsize=10)
ax.set_xlim(0.99, 1.075)
ax.set_ylim(0.985, 1.095)
ax.set_xlabel(r"Mean ratio $\beta=\mu_{pred}/\mu_{obs}$ (ideal = 1)")
ax.set_ylabel(r"Variability ratio $\alpha=\sigma_{pred}/\sigma_{obs}$ (ideal = 1)")
ax.set_title("KGE calibration plane - mean and variability fidelity")
ax.grid(True, color="0.90", linewidth=0.7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

for directory in (FIG_DIR, ARTICLE_FIG_DIR):
    fig.savefig(directory / "b0_b5_val_kge_calibration.pdf", bbox_inches="tight")
    fig.savefig(directory / "b0_b5_val_kge_calibration.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6.7))
x = np.arange(len(mse))
width = 0.68
palette = {"SB": "#cc79a7", "SDSD": "#e69f00", "LCS": "#1675ad"}
ax.bar(x, mse["SB"], width, label="SB", color=palette["SB"], edgecolor="white")
ax.bar(x, mse["SDSD"], width, bottom=mse["SB"], label="SDSD",
       color=palette["SDSD"], edgecolor="white")
ax.bar(x, mse["LCS"], width, bottom=mse["SB"] + mse["SDSD"], label="LCS",
       color=palette["LCS"], edgecolor="white")
ax.set_xticks(x, mse["configuration"], fontweight="bold")
ax.set_ylabel(r"Mean squared error (m$^2$)")
ax.set_title("Ifran GEDI VAL - MSE decomposition by scenario\ncommon unique-nearest support, 2-45 m")
ax.legend(loc="upper left", frameon=True)
ax.grid(axis="y", color="0.90", linewidth=0.7)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

kge_map = metrics.set_index("configuration")["kge_2009"]
r2_map = metrics.set_index("configuration")["r_squared"]
rows = [
    [f"{v:.2f}" for v in mse["MSE"]],
    [f"{v:.2f}" for v in mse["RMSE"]],
    [f"{r2_map[c]:.3f}" for c in mse["configuration"]],
    [f"{kge_map[c]:.3f}" for c in mse["configuration"]],
]
table = ax.table(cellText=rows,
                 rowLabels=[r"MSE (m$^2$)", "RMSE (m)", r"$r^2$", "KGE"],
                 cellLoc="center", rowLoc="center",
                 bbox=[0.0, -0.34, 1.0, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(9.5)
for cell in table.get_celld().values():
    cell.set_edgecolor("#b7bec8")
    cell.set_linewidth(0.6)

fig.text(0.5, 0.015,
         r"Common VAL support: $n=7{,}833$ GEDI shots. "
         r"$r^2$ is squared Pearson correlation; MSE = SB + SDSD + LCS.",
         ha="center", fontsize=9.5, color="0.35")
fig.subplots_adjust(left=0.10, right=0.98, top=0.88, bottom=0.31)

for directory in (FIG_DIR, ARTICLE_FIG_DIR):
    fig.savefig(directory / "b0_b5_val_mse_decomposition.pdf", bbox_inches="tight")
    fig.savefig(directory / "b0_b5_val_mse_decomposition.png", dpi=300, bbox_inches="tight")
plt.show()
